# Part 2 : Preprocessing

**Preprocessing Pipeline:**
1. **Data Loading** - Load train/test and target
2. **Drop Sparse Features** - Remove columns with >80% missing values
3. **Imputation** - Distance-weighted kNN on remaining NaNs (with temporary standardization)
4. **Regional Aggregates** - Mean across time bins per region-layer-celltype combo
5. **Session Effects** - One-hot encode `session_id` (drop first level)
6. **Feature Pruning** - Remove constant and highly correlated predictors (train only), then align test
7. **Final Scaling** - Robust scaling of pruned feature set for downstream models
8. **PCA (optional)** - Add components explaining ~95% variance on scaled features
9. **Export** - Save processed matrices, scaler, and feature names

**Necessary Imports**

In [66]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import re
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_selection import VarianceThreshold

**Data Loading**


In [67]:
# get paths
ROOT = Path.cwd().parents[0]
DATA = ROOT / "data"
train_path = DATA / "train.csv"
test_path = DATA / "test.csv"
sample_path = DATA / "sample_submission.csv"

# try loading data
try:
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    sample = pd.read_csv(sample_path)
    print("Data loaded successfully\n")
except FileNotFoundError:
    print("Error: One or more data files not found. Please check the file paths.")
    print(f"Looking for files in: {DATA}")
except pd.errors.ParserError:
    print("Error: There was an issue parsing one of the CSV files.")
except Exception as e:
    print(f"Unexpected error loading data: {e}")

# display data shapes
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"Sample submission shape: {sample.shape}")

# Reset indices to ensure alignment throughout preprocessing
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

# Separate features and target
neural_cols = [col for col in train.columns if col not in ['session_id', 'trial_number', 'TRIAL_TYPE']]
x_train = train[neural_cols].copy()
y_train = train['TRIAL_TYPE'].copy()
x_test = test[neural_cols].copy()

print(f"\nNeural features: {len(neural_cols)}")
print(f"Target classes: {y_train.nunique()}")
print(f"y_train index range: {y_train.index.min()} to {y_train.index.max()}")

Data loaded successfully

Train shape: (4407, 1383)
Test shape: (2292, 1382)
Sample submission shape: (2292, 2)

Neural features: 1380
Target classes: 10
y_train index range: 0 to 4406


**Data Cleaning**

As seen in the data inspection notebook, some features were missing significant amounts of data, we will thus remove these rows from the data set, as they don't add any information we can use for prediction later.

In [68]:
# find columns with >80% missing values
missing_fraction = x_train.isnull().mean()
high_missing_cols = missing_fraction[missing_fraction > 0.8].sort_values(ascending=False)

print(f"Features >80% missing: {len(high_missing_cols)}")
if len(high_missing_cols) > 0:
    print("Top few:")
    print(high_missing_cols.head())
    # drop from train and test
    x_train = x_train.drop(columns=high_missing_cols.index)
    x_test = x_test.drop(columns=high_missing_cols.index)

print(f"\nFeatures after dropping >80% missing: {x_train.shape[1]}")

Features >80% missing: 60
Top few:
wS2_L1_INH_time_0     1.0
wS2_L1_INH_time_15    1.0
wS2_L1_INH_time_3     1.0
wS2_L1_INH_time_4     1.0
wS2_L1_INH_time_5     1.0
dtype: float64

Features after dropping >80% missing: 1320


**Imputation and scaling (distance-weighted kNN)**

Uses `KNNImputer` with distance weighting to fill remaining missing values using nearest neighbours. However we first need to scale the data to make it behave correctly. We will also need the scaling to run PCA and linear models.

In [ ]:
# kNN imputation with pre-scaling (standardize -> impute -> invert scale)
# takes about 2 minutes to run!

missing_train = x_train.isnull().sum().sum()
missing_test = x_test.isnull().sum().sum()
print(f"Remaining missing values -> train: {missing_train}, test: {missing_test}")

# Union of columns with NaNs in train or test
cols_with_na = x_train.columns[(x_train.isnull().any()) | (x_test.isnull().any())].tolist()
print(f"Columns with NaNs (union train/test): {len(cols_with_na)}")

if (missing_train + missing_test) > 0 and cols_with_na:
    # Work only on columns that have NaNs, and standardize for distance-based kNN
    train_sub = x_train[cols_with_na].astype('float32')
    test_sub = x_test[cols_with_na].astype('float32')
    scaler_knn = StandardScaler()
    train_scaled = scaler_knn.fit_transform(train_sub)
    test_scaled = scaler_knn.transform(test_sub)

    knn_imputer = KNNImputer(n_neighbors=3, weights="distance")  # fewer neighbors for speed
    train_imputed_scaled = knn_imputer.fit_transform(train_scaled)
    test_imputed_scaled = knn_imputer.transform(test_scaled)

    # Invert scaling to original units
    train_imputed = scaler_knn.inverse_transform(train_imputed_scaled)
    test_imputed = scaler_knn.inverse_transform(test_imputed_scaled)

    # Write back into full frames
    x_train_imputed = x_train.copy()
    x_test_imputed = x_test.copy()
    x_train_imputed[cols_with_na] = train_imputed
    x_test_imputed[cols_with_na] = test_imputed

    x_train = x_train_imputed
    x_test = x_test_imputed
    print(f"Missing after kNN -> train: {x_train.isnull().sum().sum()}, test: {x_test.isnull().sum().sum()}")
elif (missing_train + missing_test) > 0:
    print("Missing values exist but no columns identified; skipping kNN.")
else:
    print("No missing values to impute; kNN skipped.")

Remaining missing values -> train: 924570, test: 446040
Columns with NaNs (union train/test): 690
Missing after kNN -> train: 0, test: 0


**Region/session-derived features**

Create region-layer-celltype aggregate features (means across time bins) and one-hot encode session IDs. This adds anatomical context and session effects as features before scaling.

In [70]:
# Aggregate neural signals by region-layer-celltype and add session one-hots
# This expands features with interpretable region summaries and session effects.

# Build mapping from column names to region-layer-celltype combos
combo_to_cols = {}
pattern = re.compile(r"([wA-Z0-9]+)_([A-Za-z0-9/]+)_(EXC|INH)_time_(\d+)")
for col in x_train.columns:
    match = pattern.match(col)
    if match:
        region = match.group(1)
        layer = match.group(2)
        cell_type = match.group(3)
        combo = f"{region}_{layer}_{cell_type}"
        combo_to_cols.setdefault(combo, []).append(col)

print(f"Detected region-layer-celltype combos: {len(combo_to_cols)}")

# Aggregate mean across time bins for each combo
agg_train = pd.DataFrame(index=x_train.index)
agg_test = pd.DataFrame(index=x_test.index)
for combo, cols in combo_to_cols.items():
    agg_train[f"agg_mean_{combo}"] = x_train[cols].mean(axis=1)
    agg_test[f"agg_mean_{combo}"] = x_test[cols].mean(axis=1)

# One-hot encode session_id (drop first to avoid full multicollinearity)
# IMPORTANT: Use reset index to ensure alignment with x_train
session_train = pd.get_dummies(train["session_id"], prefix="session", drop_first=True)
session_train = session_train.reset_index(drop=True)
session_test = pd.get_dummies(test["session_id"], prefix="session", drop_first=True)
session_test = session_test.reset_index(drop=True)

# Align session columns between train and test
session_test = session_test.reindex(columns=session_train.columns, fill_value=0)

# Ensure all DataFrames have consistent indices before concatenation
x_train = x_train.reset_index(drop=True)
x_test = x_test.reset_index(drop=True)
agg_train = agg_train.reset_index(drop=True)
agg_test = agg_test.reset_index(drop=True)

# Concatenate enriched features
x_train = pd.concat([x_train, agg_train, session_train], axis=1)
x_test = pd.concat([x_test, agg_test, session_test], axis=1)

print(f"x_train with region/session features: {x_train.shape}")
print(f"x_test with region/session features: {x_test.shape}")

Detected region-layer-celltype combos: 44
x_train with region/session features: (4407, 1371)
x_test with region/session features: (2292, 1371)


**Feature Pruning (post-aggregation)**

Drop zero-variance and highly correlated features using the enriched train set; apply the kept columns to test to avoid leakage.

In [71]:
# Remove constant predictors (zero variance) and highly correlated predictors after feature creation
# takes about 15 seconds to run!

# Constant features
variance = x_train.var()
constant_features = variance[variance == 0].index.tolist()
print(f"Constant features (zero variance): {len(constant_features)}")

if constant_features:
    x_train = x_train.drop(columns=constant_features)
    x_test = x_test.drop(columns=constant_features)

# Correlated features (train only)
correlation_matrix = x_train.corr().abs()
upper_triangle = np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
upper_corr = correlation_matrix.where(upper_triangle)

high_corr_features = [col for col in upper_corr.columns if any(upper_corr[col] > 0.95)]
print(f"Highly correlated features (r > 0.95): {len(high_corr_features)}")

if high_corr_features:
    x_train = x_train.drop(columns=high_corr_features)
    x_test = x_test.drop(columns=high_corr_features)

print(f"\nFeatures after pruning: {x_train.shape[1]}")

Constant features (zero variance): 0
Highly correlated features (r > 0.95): 0

Features after pruning: 1371


**Standardization**

Apply RobustScaler to the pruned, enriched feature set (original signals + region aggregates + session one-hots) before modeling.

In [73]:
# Standardization with handling of near-zero IQR columns
# RobustScaler can produce extreme values when IQR ≈ 0

# Convert any boolean columns to int (from session one-hots)
bool_cols = x_train.select_dtypes(include=['bool']).columns.tolist()
if bool_cols:
    x_train[bool_cols] = x_train[bool_cols].astype(int)
    x_test[bool_cols] = x_test[bool_cols].astype(int)
    print(f"Converted {len(bool_cols)} boolean columns to int")

# First, identify columns with near-zero IQR (causes scaling issues)
# Only check numeric columns
numeric_cols = x_train.select_dtypes(include=[np.number]).columns
iqr = x_train[numeric_cols].quantile(0.75) - x_train[numeric_cols].quantile(0.25)
near_zero_iqr = iqr[iqr < 1e-6].index.tolist()
print(f"Columns with near-zero IQR: {len(near_zero_iqr)}")

if near_zero_iqr:
    # Drop these columns as they're essentially constant and cause scaling issues
    x_train = x_train.drop(columns=near_zero_iqr)
    x_test = x_test.drop(columns=near_zero_iqr)
    print(f"Dropped {len(near_zero_iqr)} near-zero IQR columns")

# Now apply RobustScaler
scaler = RobustScaler()

x_train_scaled = pd.DataFrame(
    scaler.fit_transform(x_train),
    columns=x_train.columns,
    index=x_train.index
)
x_test_scaled = pd.DataFrame(
    scaler.transform(x_test),
    columns=x_test.columns,
    index=x_test.index
)

# Verify no extreme values
max_val = x_train_scaled.abs().max().max()
print(f"\nScaled data shape: {x_train_scaled.shape}")
print(f"Max absolute value after scaling: {max_val:.2f}")
if max_val > 100:
    print("WARNING: Some values are still extreme after scaling!")
print(f"Median (should be ~0): {x_train_scaled.median().median():.4f}")
print(f"IQR (should be ~1): {(x_train_scaled.quantile(0.75) - x_train_scaled.quantile(0.25)).median():.4f}")

Converted 7 boolean columns to int
Columns with near-zero IQR: 59
Dropped 59 near-zero IQR columns

Scaled data shape: (4407, 1312)
Max absolute value after scaling: 40.00
Median (should be ~0): 0.0000
IQR (should be ~1): 1.0000


**PCA**

Fit PCA on the scaled features to capture ~95% variance; append components to the scaled dataset for modeling.

In [74]:
# Fit PCA on scaled features (keep ~95% variance) and append components
pca = PCA(n_components=0.95, random_state=42)

pca_train = pca.fit_transform(x_train_scaled)
pca_test = pca.transform(x_test_scaled)

pca_cols = [f"PCA_{i+1}" for i in range(pca.n_components_)]
pca_train_df = pd.DataFrame(pca_train, columns=pca_cols, index=x_train_scaled.index)
pca_test_df = pd.DataFrame(pca_test, columns=pca_cols, index=x_test_scaled.index)

# Concatenate PCA components to scaled features
x_train_final = pd.concat([x_train_scaled, pca_train_df], axis=1)
x_test_final = pd.concat([x_test_scaled, pca_test_df], axis=1)

print(f"PCA components: {pca.n_components_}")
print(f"Cumulative variance explained: {pca.explained_variance_ratio_.sum():.3f}")
print(f"Final feature counts (with PCA): train={x_train_final.shape}, test={x_test_final.shape}")

PCA components: 804
Cumulative variance explained: 0.950
Final feature counts (with PCA): train=(4407, 2116), test=(2292, 2116)


**Post-PCA pruning (optional)**

Remove highly correlated features after PCA augmentation to slim the final matrix.

In [ ]:
# Optional: prune highly correlated features after PCA augmentation
# Takes about 40 seconds to run!

corr_matrix_final = x_train_final.corr().abs()
upper_triangle_final = np.triu(np.ones(corr_matrix_final.shape), k=1).astype(bool)
upper_corr_final = corr_matrix_final.where(upper_triangle_final)

high_corr_final = [col for col in upper_corr_final.columns if any(upper_corr_final[col] > 0.95)]
print(f"Post-PCA highly correlated features (r > 0.95): {len(high_corr_final)}")

if high_corr_final:
    x_train_final = x_train_final.drop(columns=high_corr_final)
    x_test_final = x_test_final.drop(columns=high_corr_final)

print(f"Final shapes after post-PCA pruning: train={x_train_final.shape}, test={x_test_final.shape}")

Post-PCA highly correlated features (r > 0.95): 0
Final shapes after post-PCA pruning: train=(4407, 2116), test=(2292, 2116)


**Export Preprocessed Data**

Save the cleaned, enriched, scaled data with PCA components for use in modeling notebooks.

In [76]:
# Create processed data directory
# takes 15 seconds to save!

processed_dir = DATA / "processed"
processed_dir.mkdir(exist_ok=True)

# Remove existing files in processed_dir to ensure fresh outputs
for f in processed_dir.glob("*"):
    if f.is_file():
        f.unlink()

# CRITICAL: Reset indices and ensure y_train aligns with x_train_final
x_train_final = x_train_final.reset_index(drop=True)
x_test_final = x_test_final.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)

# Verify alignment
assert len(x_train_final) == len(y_train), f"Row count mismatch: x={len(x_train_final)}, y={len(y_train)}"
print(f"Alignment check passed: {len(x_train_final)} rows in both x_train and y_train")

# Save preprocessed data (with PCA components)
x_train_final.to_csv(processed_dir / "x_train_scaled.csv", index=False)
x_test_final.to_csv(processed_dir / "x_test_scaled.csv", index=False)
y_train.to_csv(processed_dir / "y_train.csv", index=False)

# Also save the scaler and PCA model and feature names for later use
import joblib
joblib.dump({
    "scaler": scaler,
    "pca": pca
}, processed_dir / "transforms.joblib")

# Save feature names
with open(processed_dir / "feature_names.txt", "w") as f:
    f.write("\n".join(x_train_final.columns))

print(f"Preprocessed data saved to: {processed_dir}")
print(f"\nFiles created:")
print(f"  - x_train_scaled.csv: {x_train_final.shape}")
print(f"  - x_test_scaled.csv: {x_test_final.shape}")
print(f"  - y_train.csv: {y_train.shape}")
print(f"  - transforms.joblib (scaler + pca)")
print(f"  - feature_names.txt")

Alignment check passed: 4407 rows in both x_train and y_train
Preprocessed data saved to: c:\Users\milor\Projects\ML-BIO-322\BIO-322\data\processed

Files created:
  - x_train_scaled.csv: (4407, 2116)
  - x_test_scaled.csv: (2292, 2116)
  - y_train.csv: (4407,)
  - transforms.joblib (scaler + pca)
  - feature_names.txt
